[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onuralpszr/litert-llm-cookbook/blob/main/colab/11_openai_api_server.ipynb)

# Example 11: Local API Server (OpenAI and Gemini)

LiteRT-LM can serve a local HTTP API that speaks either the OpenAI Responses API
or the Gemini API. This notebook starts the server as a background subprocess
and then runs client requests against it.

The setup order matters:

1. Install dependencies
2. Download and import the model into the LiteRT-LM local store
3. Start the server in a background cell
4. Run the client cells
5. Shut the server down when you are done

Only one server mode can be active at a time. The cells below default to the
OpenAI Responses API. Scroll to the Gemini section to switch modes.

In [ ]:
!pip install litert-lm-api-nightly litert-lm openai requests -q

In [ ]:
# Download the model file
import subprocess
subprocess.run([
    "curl", "-L",
    "https://huggingface.co/litert-community/gemma-4-E2B-it-litert-lm/resolve/main/gemma-4-E2B-it.litertlm?download=true",
    "-o", "/content/gemma-4-E2B-it.litertlm"
], check=True)

In [ ]:
# Import the model into the LiteRT-LM local store.
# The serve command looks up models here, not from a bare file path.
import subprocess
result = subprocess.run(
    [
        "litert-lm", "import",
        "--from-huggingface-repo=litert-community/gemma-4-E2B-it-litert-lm",
        "gemma-4-E2B-it.litertlm",
    ],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

## Start the server

Run one of the two cells below depending on which API you want to use.
Do not run both at the same time.

In [ ]:
# Option A: OpenAI Responses API (POST /v1/responses)
import subprocess, time
server = subprocess.Popen(
    ["litert-lm", "serve", "--api", "openai", "--host", "localhost", "--port", "9379"]
)
time.sleep(5)
print("Server started (OpenAI Responses API). PID:", server.pid)

In [ ]:
# Option B: Gemini API (POST /v1beta/models/{id}:generateContent)
# Uncomment this cell and comment out Option A if you want the Gemini API instead.
# import subprocess, time
# server = subprocess.Popen(
#     ["litert-lm", "serve", "--api", "gemini", "--host", "localhost", "--port", "9379"]
# )
# time.sleep(5)
# print("Server started (Gemini API). PID:", server.pid)

## A. OpenAI Responses API

Run these cells when the server was started with `--api openai`.

In [ ]:
# A1: Basic call via the OpenAI Python SDK
from openai import OpenAI

HOST = "http://localhost:9379"
MODEL_ID = "gemma-4-E2B-it.litertlm"

openai_client = OpenAI(base_url=f"{HOST}/v1", api_key="not-needed")

response = openai_client.responses.create(
    model=MODEL_ID,
    input="What is the capital of Turkey?",
)
print(response.output_text)

In [ ]:
# A2: Streaming via raw SSE
import json, requests

HOST = "http://localhost:9379"
MODEL_ID = "gemma-4-E2B-it.litertlm"

payload = {
    "model": MODEL_ID,
    "input": "Write a short haiku about the Black Sea.",
    "stream": True,
}
event_type = None
with requests.post(f"{HOST}/v1/responses", json=payload, stream=True, timeout=60) as resp:
    resp.raise_for_status()
    for line in resp.iter_lines():
        if not line:
            event_type = None
            continue
        text = line.decode("utf-8")
        if text.startswith("event: "):
            event_type = text[7:]
        elif text.startswith("data: "):
            raw = text[6:]
            if raw == "[DONE]":
                break
            if event_type == "response.output_text.delta":
                try:
                    chunk = json.loads(raw)
                    delta = chunk.get("delta", {}).get("text", "")
                    if delta:
                        print(delta, end="", flush=True)
                except json.JSONDecodeError:
                    pass
print()

In [ ]:
# A3: Raw HTTP POST without the SDK
import json, requests

HOST = "http://localhost:9379"
MODEL_ID = "gemma-4-E2B-it.litertlm"

payload = {"model": MODEL_ID, "input": "Explain on-device AI in one sentence."}
resp = requests.post(f"{HOST}/v1/responses", json=payload, timeout=60)
resp.raise_for_status()
data = resp.json()
print(data["output"][0]["content"][0]["text"])

In [ ]:
# A4: Raw HTTP streaming
import json, requests

HOST = "http://localhost:9379"
MODEL_ID = "gemma-4-E2B-it.litertlm"

payload = {"model": MODEL_ID, "input": "Name three things Turkey is famous for.", "stream": True}
with requests.post(f"{HOST}/v1/responses", json=payload, stream=True, timeout=60) as resp:
    resp.raise_for_status()
    for line in resp.iter_lines():
        if not line:
            continue
        text = line.decode("utf-8")
        if text.startswith("data: "):
            raw = text[6:]
            if raw == "[DONE]":
                break
            try:
                chunk = json.loads(raw)
                delta = chunk.get("delta", {}).get("text", "")
                if delta:
                    print(delta, end="", flush=True)
            except json.JSONDecodeError:
                pass
print()

## B. Gemini API

Run these cells when the server was started with `--api gemini`.
Restart the server with the Gemini option before running this section.

In [ ]:
# B1: Basic generateContent
import json, requests

HOST = "http://localhost:9379"
MODEL_ID = "gemma-4-E2B-it.litertlm"
GEMINI_BASE = f"{HOST}/v1beta/models/{MODEL_ID}"

payload = {
    "contents": [
        {"role": "user", "parts": [{"text": "What is the capital of Turkey?"}]}
    ]
}
resp = requests.post(f"{GEMINI_BASE}:generateContent", json=payload, timeout=60)
resp.raise_for_status()
data = resp.json()
print(data["candidates"][0]["content"]["parts"][0]["text"])

In [ ]:
# B2: generateContent with a system instruction
import json, requests

HOST = "http://localhost:9379"
MODEL_ID = "gemma-4-E2B-it.litertlm"
GEMINI_BASE = f"{HOST}/v1beta/models/{MODEL_ID}"

payload = {
    "systemInstruction": {
        "parts": [{"text": "You are a concise geography expert. Keep answers under two sentences."}]
    },
    "contents": [
        {"role": "user", "parts": [{"text": "Tell me about Trabzon."}]}
    ],
}
resp = requests.post(f"{GEMINI_BASE}:generateContent", json=payload, timeout=60)
resp.raise_for_status()
data = resp.json()
print(data["candidates"][0]["content"]["parts"][0]["text"])

In [ ]:
# B3: Multi-turn conversation (history passed in the request body)
import json, requests

HOST = "http://localhost:9379"
MODEL_ID = "gemma-4-E2B-it.litertlm"
GEMINI_BASE = f"{HOST}/v1beta/models/{MODEL_ID}"

payload = {
    "contents": [
        {"role": "user", "parts": [{"text": "Name one city on the Black Sea coast of Turkey."}]},
        {"role": "model", "parts": [{"text": "Trabzon is a major city on the Black Sea coast."}]},
        {"role": "user", "parts": [{"text": "What is that city known for?"}]},
    ]
}
resp = requests.post(f"{GEMINI_BASE}:generateContent", json=payload, timeout=60)
resp.raise_for_status()
data = resp.json()
print(data["candidates"][0]["content"]["parts"][0]["text"])

In [ ]:
# B4: Streaming via streamGenerateContent
import json, requests

HOST = "http://localhost:9379"
MODEL_ID = "gemma-4-E2B-it.litertlm"
GEMINI_BASE = f"{HOST}/v1beta/models/{MODEL_ID}"

payload = {
    "contents": [
        {"role": "user", "parts": [{"text": "Write a short poem about on-device AI."}]}
    ]
}
with requests.post(
    f"{GEMINI_BASE}:streamGenerateContent",
    json=payload, stream=True, timeout=60,
) as resp:
    resp.raise_for_status()
    for line in resp.iter_lines():
        if not line:
            continue
        text = line.decode("utf-8")
        if text.startswith("data: "):
            raw = text[6:]
            try:
                chunk = json.loads(raw)
                for part in chunk.get("candidates", [{}])[0].get("content", {}).get("parts", []):
                    t = part.get("text", "")
                    if t:
                        print(t, end="", flush=True)
            except json.JSONDecodeError:
                pass
print()

## Shut down the server

Run this cell when you are finished.

In [ ]:
server.terminate()
print("Server stopped.")